In [52]:
import os
import argparse
import pandas as pd
import numpy as np
from tqdm import tqdm
import itertools
from multiprocessing import Pool

In [53]:
# Data Options
identity_labels_path = "/home/nthom/Documents/datasets/CelebA/Anno/identity_CelebA.csv"
attribute_labels_path = "/home/nthom/Documents/datasets/CelebA/Anno/list_attr_celeba.txt"

# Preprocessing Options
preproc_save_path = "./preprocessed_data/"
under_represented_threshold = 29

In [54]:
# Read in data frames
identity_labels_df = pd.read_csv(identity_labels_path)
attribute_labels_df = pd.read_csv(attribute_labels_path, sep=" ", skiprows=1)

In [55]:
identity_counts_dict = {}
under_represented_identities_dict = {}

max = 0
min = np.inf

print("Calculating Dataset Statistics:")
for identity in tqdm(identity_labels_df.identity_id.unique()):
    count = len(identity_labels_df[identity_labels_df["identity_id"] == identity].index)

    if count > max:
        max = count
    if count < min:
        min = count

    if count <= under_represented_threshold:
        under_represented_identities_dict[identity] = count
    identity_counts_dict[identity] = count

Calculating Dataset Statistics:


100%|██████████| 10177/10177 [00:03<00:00, 3362.12it/s]


In [56]:
print(f"Max: {max}, Min: {min}")
print(f"Under Represented Identities Threshold: {under_represented_threshold} samples")
print(f"Num Under Represented Identities: {len(list(under_represented_identities_dict.keys()))}, Num Identities: {len(list(identity_counts_dict.keys()))}")

Max: 35, Min: 1
Under Represented Identities Threshold: 29 samples
Num Under Represented Identities: 7817, Num Identities: 10177


In [57]:
for identity_name in tqdm(under_represented_identities_dict.keys()):
    identity_image_names = identity_labels_df[identity_labels_df["identity_id"] == identity_name]["image_name"].values.tolist()
    attribute_labels_df = attribute_labels_df[~attribute_labels_df.image_name.isin(identity_image_names)]

identity_labels_df = identity_labels_df[~identity_labels_df.identity_id.isin(list(under_represented_identities_dict.keys()))]

100%|██████████| 7817/7817 [02:20<00:00, 55.75it/s] 


In [58]:
identity_counts_dict = {}
under_represented_identities_dict = {}

max = 0
min = np.inf

print("Calculating Dataset Statistics:")
for identity in tqdm(identity_labels_df.identity_id.unique()):
    count = len(identity_labels_df[identity_labels_df["identity_id"] == identity].index)

    if count > max:
        max = count
    if count < min:
        min = count

    if count <= under_represented_threshold:
        under_represented_identities_dict[identity] = count
    identity_counts_dict[identity] = count

Calculating Dataset Statistics:


100%|██████████| 2360/2360 [00:00<00:00, 4412.18it/s]


In [59]:
print(f"Max: {max}, Min: {min}")
print(f"Under Represented Identities (under {under_represented_threshold} samples):")
print(f"Num Under Represented Identities: {len(list(under_represented_identities_dict.keys()))}, Num Identities: {len(list(identity_counts_dict.keys()))}")

Max: 35, Min: 30
Under Represented Identities (under 29 samples):
Num Under Represented Identities: 0, Num Identities: 2360


In [60]:
identity_labels_df.to_csv(f"./preprocessed_data/pruned_by_num_samples/identity_CelebA_min-{under_represented_threshold+1}.csv")
attribute_labels_df.to_csv(f"./preprocessed_data/pruned_by_num_samples/list_attr_celeba_min-{under_represented_threshold+1}.csv")